# Q1 — Infinite reactor of pure U-238
$$k_\infty=\frac{\nu\times\text{fissions}}{\text{source neutrons}},\qquad \nu=2.42$$
Pure $^{238}$U. Infinite → no leakage, so only atom *ratios* matter; absolute density cancels.

### 1. Setup
Constants.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, urllib.request, re, os, time
plt.rcParams.update({"figure.dpi":120,"font.size":9,"axes.grid":True,"grid.alpha":.3})

SEED   = 20260908
NU     = 2.42             # neutrons per fission
T_FISS = 1/0.776          # MeV, Maxwellian fission spectrum
KT     = 2.53e-8          # MeV, room temperature (0.0253 eV)
E_TH   = 1e-7             # MeV, below this the neutron is thermal
E_MIN  = 1e-10            # MeV, floor of every cross section table
print(f"nu={NU}   T_fiss={T_FISS:.4f} MeV   thermal cutoff {E_TH*1e6:.2f} eV")

### 2. Energy grid
Cross sections are also tabulated on a fixed grid, $(0.2,0.4,0.6,0.8,1.0)\times10^{y}$ MeV, $y=-8\ldots1$ — 50 points, five per decade.

In [ ]:
E_GRID = np.sort((np.array([.2,.4,.6,.8,1.])[None,:] * 10.0**np.arange(-8,2)[:,None]).ravel())
print(f"{E_GRID.size} points: {E_GRID[0]:.1e} MeV ({E_GRID[0]*1e6:.3f} eV) .. {E_GRID[-1]:.1e} MeV")

### 3. Cross sections from KAERI
Pointwise ENDF/B-VIII.0. MT = 2 elastic, 4 inelastic, 102 capture, 18 fission.

In [ ]:
KAERI = "https://atom.kaeri.re.kr/nuchart/getData.jsp?target=endfb8.0,{z},{a},{mat},3,{mt}"
ZAM   = {"U8":(92,238,9237)}
MTN   = {"e":2, "i":4, "c":102, "f":18}
HAVE  = {"U8":"eicf"}          # which reactions each nuclide actually has

def fetch(nuc, mt):
    z,a,mat = ZAM[nuc]; f = f"kaeri_{nuc}_{mt}.txt"
    if not os.path.exists(f):
        with urllib.request.urlopen(KAERI.format(z=z,a=a,mat=mat,mt=mt), timeout=300) as r:
            tmp = f"{f}.{os.getpid()}.part"     # write, then rename: a parallel run never reads a half-written file
            open(tmp,"w").write(re.sub(r"<[^>]*>", "", r.read().decode("utf-8","replace")))
            os.replace(tmp, f)
    E, S = [], []
    for ln in open(f):
        p = ln.split()
        if len(p) == 2:
            try: E.append(float(p[0])); S.append(float(p[1]))
            except ValueError: pass
    E, S = np.array(E)*1e-6, np.array(S)          # eV -> MeV
    o = np.argsort(E, kind="stable")
    return E[o], S[o]

t = time.time()
RAW = {q:{k:fetch(q, MTN[k]) for k in HAVE[q]} for q in ZAM}
for q in RAW:
    for k,(E,S) in RAW[q].items():
        print(f"{q:3s} MT={MTN[k]:3d} {k}: {E.size:7,} pts, {E[S>0][0]:.1e}..{E[-1]:.1e} MeV, peak {S.max():.4g} b")
print(f"[{time.time()-t:.0f}s]")

### 4. Reading a value off the curve
Straight line in $(\log E,\ \log\sigma)$ between the two nearest tabulated points.
The transport uses the **full pointwise** data; the 50-point table is written to a CSV
file for reference (section 17 shows what using only the coarse grid would cost).

In [ ]:
class XSec:
    def __init__(self, E, S):
        p = S > 0
        self.E0, self.E1, self.S1 = E[p][0], E[-1], S[-1]
        self.lE, self.lS = np.log(E[p]), np.log(S[p])
    def __call__(self, E):
        E = np.atleast_1d(np.asarray(E, float)); out = np.zeros_like(E)
        m = (E >= self.E0) & (E <= self.E1)
        out[m] = np.exp(np.interp(np.log(E[m]), self.lE, self.lS))
        out[E > self.E1] = self.S1
        return out

ZERO = lambda E: np.zeros_like(np.atleast_1d(np.asarray(E, float)))
XS   = {q:{k:(XSec(*RAW[q][k]) if k in HAVE[q] else ZERO) for k in "eicf"} for q in ZAM}

# required tabulation, one CSV per nuclide
for q in ZAM:
    np.savetxt(f"xs_{q}_Q1.csv", np.column_stack([E_GRID]+[XS[q][k](E_GRID) for k in "eicf"]),
               delimiter=",", fmt="%.6g", header="E_MeV,sigma_e,sigma_i,sigma_c,sigma_f", comments="")
TH = 2.53e-8
print("thermal (0.0253 eV) spot checks against published values:")

for q in ZAM:
    print(f"  {q:3s} " + "  ".join(f"sigma_{k}={XS[q][k](TH)[0]:9.4f} b" for k in HAVE[q]))
print("  refs: U5 sf 585, sc 99 | U8 sc 2.68 | H sc 0.3326 | D sc 0.000519 | O se 3.85")

### 5. Plot 1 — cross sections

In [ ]:
Ep  = np.logspace(np.log10(E_GRID[0]), np.log10(E_GRID[-1]), 3000)
lab = {"e":"elastic","i":"inelastic","c":"capture","f":"fission"}
fig, ax = plt.subplots(1, 1, figsize=(4.0, 3.5), squeeze=False)
for a,q in zip(ax[0], ZAM):
    for k in HAVE[q]:
        a.loglog(Ep, np.maximum(XS[q][k](Ep),1e-12), lw=.8, label=lab[k])
        a.loglog(E_GRID, np.maximum(XS[q][k](E_GRID),1e-12), ".", ms=3, alpha=.5)
    a.set(title=q, xlabel="E (MeV)", ylabel="$\\sigma$ (b)", ylim=(1e-5,1e5)); a.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("dots are the 50 tabulated points; lines are the full pointwise data the transport uses")

### 6. Composition
Fixed by the problem — nothing to choose.

In [ ]:
AMASS = {"U8":238.0}
ALPHA = {q:((A-1)/(A+1))**2 for q,A in AMASS.items()}

COMP = {"U8": 1.0}
print("relative atom fractions (absolute scale cancels in an infinite medium):")
for q,v in COMP.items(): print(f"  {q:3s} {v:12.6f}")

### 7. Neutron birth and source spectrum (step 1)
Every history begins by **birthing one neutron**, which fixes its energy (step 1). This is where
the neutron counter goes up by one. A surviving neutron never comes back here: it goes to step 2
with its new energy. Only a non-surviving neutron (capture or fission) sends the procedure back to
step 1, and only while the total number of neutrons has not been reached (step 6).

The birth energy is sampled from this fission spectrum:
$$s(E)=0.771\,\sqrt{E}\;e^{-0.776E}\qquad(E\text{ in MeV})$$
This is exactly a Gamma(3/2, T) distribution with $T = 1/0.776 = 1.2887$ MeV, so it can be sampled
directly with no rejection. Mean birth energy $3T/2 = 1.933$ MeV.

Only the **energy** is sampled. In an infinite homogeneous medium there is no leakage, so a
neutron's birth position and direction have no effect on what happens to it.

In [ ]:
S_analytic   = lambda E: 0.771*np.sqrt(E)*np.exp(-0.776*E)
sample_birth = lambda m, r: r.gamma(1.5, T_FISS, size=m)
print(f"T = {T_FISS:.4f} MeV   mean 3T/2 = {1.5*T_FISS:.4f} MeV   normalisation "
      f"2/(sqrt(pi) T^1.5) = {2/(np.sqrt(np.pi)*T_FISS**1.5):.4f} (quoted 0.771)")

### 8. Plot 2 — sampler check

In [ ]:
r0 = np.random.default_rng(SEED)
Eb = sample_birth(500_000, r0)
b  = np.linspace(0,12,200); mid = .5*(b[1:]+b[:-1])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for a, sc in zip(ax, ("linear","log")):
    a.hist(Eb, bins=b, density=True, histtype="step", lw=1.4, label="sampled")
    a.plot(mid, S_analytic(mid), "k:", lw=2, label="$s(E)$")
    a.set(xlabel="E (MeV)", ylabel="pdf", yscale=sc)
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"sampled mean {Eb.mean():.4f} MeV vs exact {1.5*T_FISS:.4f}   P(E>1 MeV) = {np.mean(Eb>1):.4f}")

### 9. Scattering and the thermal cutoff (steps 4–5)
Elastic: $E'=E[\alpha+(1-\alpha)\xi]$ with $\alpha=\left(\frac{A-1}{A+1}\right)^2$.
Inelastic: subtract a sampled nuclear level.
Below $E_{\rm th}=0.1$ eV the neutron is in equilibrium with the medium, so its energy is
resampled from a Maxwellian at 293 K rather than slowing down forever.

In [ ]:
LEVELS = {"U8":np.array([.0449,.1483,.3072,.5182,.6801,.7318,.8271,.9302,.9663,1.0374,1.0607,1.1259,1.1685])}

def elastic(q, E, r):
    a = ALPHA[q]
    return E*(a + (1-a)*r.random(E.size))

def inelastic(q, E, r):
    L = LEVELS[q]
    k = np.maximum(np.searchsorted(L, E), 1)
    return np.maximum(E - L[(r.random(E.size)*k).astype(np.int64)], E_MIN)

r0 = np.random.default_rng(0); E2 = np.full(200_000, 2.0)
for q in ZAM:
    xi = 1.0 if ALPHA[q]==0 else 1 + ALPHA[q]*np.log(ALPHA[q])/(1-ALPHA[q])
    print(f"{q:3s} A={AMASS[q]:6.1f}  alpha={ALPHA[q]:.5f}  xi={xi:.5f}  "
          f"mean E' after one elastic collision at 2 MeV = {elastic(q,E2,r0).mean():.4f} MeV  "
          f"({np.log(2e6/0.0253)/xi:.0f} collisions to thermalise)")

#### Modelling assumption: energy loss in inelastic scattering
The KAERI data used here are ENDF/B-VIII.0 **cross sections** (file MF3). For inelastic scattering
(MT=4) that gives *how often* it happens, not *how much energy* the neutron loses. So the energy
loss has to be modelled.

**Model:** the struck nucleus is left in one of its lowest excited levels, chosen uniformly from the
levels lying below the neutron's energy $E$. The neutron leaves with $E - E_{\text{level}}$.
- $^{238}$U: 13 levels, 44.9 keV to 1.1685 MeV (average loss about 0.73 MeV per inelastic collision)

**What this does to a fast neutron:** it loses several hundred keV per inelastic collision. A neutron
born near the mean 1.93 MeV therefore falls below the ~1 MeV $^{238}$U fission threshold within
**one or two** inelastic collisions. It does not always happen in one: from 2 MeV only the 4 levels
above 1 MeV (about 31% of draws) take it below threshold at once.

**Why the verdict is safe despite the simplification:** the $^{238}$U level list stops at 1.17 MeV,
and the real nucleus also has a continuum of higher levels. So at high energy this model
*underestimates* the energy lost, which keeps neutrons above the fission threshold longer and pushes
$k_\infty$ **up**. The true $k_\infty$ is, if anything, lower than computed here, so "cannot go
critical" holds.

### 10. Transport — the collision procedure, step by step
One neutron at a time, following seven steps:

1. **Neutron energy fixed** — birth energy sampled from $s(E)$.
2. **Colliding nucleus fixed** — nuclide $i$ chosen with probability $N_i\sigma_{t,i}/\sum_j N_j\sigma_{t,j}$, using the cross sections at the current energy.
3. **Type of interaction fixed** — elastic, inelastic, capture or fission, with probability $\sigma_x/\sigma_t$ for the struck nuclide.
4. **Fate of neutron fixed** — elastic/inelastic: survives. Capture/fission: does not survive.
5. **If surviving:** new energy fixed (section 9), collision counted, **go to step 2**.
6. **If not surviving:** fission or capture counted, **go to step 1** if the total number of neutrons is not exceeded.
7. **Show statistics and draw conclusion** — sections 11–18.

Infinite medium, so there is no leakage step.

In [ ]:
def run_analog(n_hist, seed=SEED):
    r = np.random.default_rng(seed)
    names = list(COMP); cum_n = np.cumsum([COMP[q] for q in names])
    fis = 0; nc_all = []
    for _ in range(n_hist):
        E, nc = r.gamma(1.5, T_FISS), 0                          # (1) birth: neutron energy fixed
        while True:                                              # (2) new collision at energy E
            s   = {q:{k: COMP[q]*XS[q][k](E)[0] for k in "eicf"} for q in names}
            stn = np.array([sum(s[q].values()) for q in names])
            q   = names[int(np.searchsorted(np.cumsum(stn), r.random()*stn.sum()))]  # (2) colliding nucleus fixed
            se, si, sc, sf = (s[q][k] for k in "eicf")
            xi = r.random()*(se+si+sc+sf); nc += 1                # (3) type of interaction fixed
            if   xi < se:        E = elastic(q, np.array([E]), r)[0]        # (4) fate: scatter survives -> (5) new energy
            elif xi < se+si:     E = inelastic(q, np.array([E]), r)[0]
            elif xi < se+si+sc:  break                            # (4)(6) capture: not surviving, history ends
            else:                fis += 1; break                 # (4)(6) fission: counted, history ends
            if E < E_TH: E = max(r.gamma(1.5, KT), E_MIN)         # thermal equilibrium
        nc_all.append(nc)
    return dict(n=n_hist, fissions=fis, k=NU*fis/n_hist, nc=np.array(nc_all))

t = time.time(); slow = run_analog(2_000)
print(f"analog loop: N={slow['n']:,}  k_inf={slow['k']:.4f}  "
      f"collisions/history={slow['nc'].mean():.1f}  [{time.time()-t:.0f}s]")

### 11. Vectorised — one million neutrons
Same physics, whole population stepped together; finished histories are compacted out.

In [ ]:
N_U, N_H2O, N_D2O = 0.04820, 0.03336, 0.03323   # atoms or molecules per barn-cm: U metal, H2O, D2O
LE   = np.linspace(-10, np.log10(20.0), 141)   # log10 E (MeV) bins for the tallies
CMAX = 3000                                     # collisions recorded in the slowing-down map

def n_scale(comp):
    """atoms/(b cm) per unit of the relative fractions: uranium metal + water, volumes additive.
    Only the TIME tallies use this; k_inf does not depend on it."""
    vol = (comp.get("U5",0)+comp.get("U8",0))/N_U + comp.get("O",0)/(N_D2O if "D" in comp else N_H2O)
    return 1/vol

def run(n_hist, comp=None, seed=SEED, batch=250_000, maxcol=200_000, tally=True):
    comp = comp or COMP
    names = list(comp); r = np.random.default_rng(seed)
    rt = np.random.default_rng(seed + 10**6)      # separate stream for flight lengths: k is unaffected
    fis = 0; ncoll = 0; done = 0; fE = []; ncs = []
    nb = LE.size-1; scale = n_scale(comp)
    smap = np.zeros((CMAX, nb)); flux = np.zeros(nb); tF, tS, tT = [], [], []
    absE = {(q,k): np.zeros(nb) for q in names for k in "cf"}
    expF, expC = np.zeros(nb), np.zeros(nb)       # fuel fission/capture probability summed over collisions
    while done < n_hist:
        m = int(min(batch, n_hist-done)); done += m
        E = np.minimum(r.gamma(1.5, T_FISS, m), 20.0)             # (1) birth
        nc = np.zeros(m, np.int64); it = 0
        tf, ts, tt = np.zeros(m), np.zeros(m), np.zeros(m)   # time spent fast / slowing down / thermal
        while E.size and it < maxcol:
            it += 1; ncoll += E.size; nc += 1                      # (2) collision
            s   = {q:{k: comp[q]*XS[q][k](E) for k in "eicf"} for q in names}
            stn = {q: sum(s[q].values()) for q in names}
            tot = np.maximum(sum(stn.values()), 1e-30)
            cum = np.cumsum([stn[q] for q in names], axis=0)/tot   # (2) nucleus
            pick = (r.random(E.size)[None,:] > cum).sum(axis=0)
            if tally:
                ib  = np.clip(np.searchsorted(LE, np.log10(E))-1, 0, nb-1)
                if it <= CMAX: smap[it-1] += np.bincount(ib, minlength=nb)
                lam = 1/(tot*scale)                          # mean free path (cm)
                flux += np.bincount(ib, lam, minlength=nb)   # collision estimator of the flux
                fu = [q for q in names if q in ("U5","U8")]
                expF += np.bincount(ib, sum(s[q]["f"] for q in fu)/tot, minlength=nb)
                expC += np.bincount(ib, sum(s[q]["c"] for q in fu)/tot, minlength=nb)
                dt  = rt.exponential(lam)/(1.383e9*np.sqrt(E))   # sampled flight time to this collision (s)
                tf += np.where(E > 0.1, dt, 0)
                ts += np.where((E <= 0.1) & (E > 1e-6), dt, 0)
                tt += np.where(E <= 1e-6, dt, 0)
            newE = E.copy(); die = np.zeros(E.size, bool)
            for j,q in enumerate(names):
                mq = pick == j
                if not mq.any(): continue
                se, si, sc, sf = (s[q][k][mq] for k in "eicf")
                xi = r.random(int(mq.sum()))*(se+si+sc+sf)         # (3) interaction
                ie  = xi < se
                ii  = ~ie & (xi < se+si)
                ic  = ~ie & ~ii & (xi < se+si+sc)
                iff = ~(ie|ii|ic)
                Em = E[mq]; Nm = Em.copy()                         # (4)-(5) fate, new energy
                if ie.any(): Nm[ie] = elastic(q, Em[ie], r)
                if ii.any(): Nm[ii] = inelastic(q, Em[ii], r)
                if iff.any():
                    fis += int(iff.sum())
                    if tally: fE.append(Em[iff])
                if tally:
                    for k,hit in (("c",ic), ("f",iff)):
                        if hit.any(): absE[q,k] += np.bincount(ib[mq][hit], minlength=nb)
                newE[mq] = Nm; die[mq] = ic|iff
            th = newE < E_TH
            if th.any(): newE[th] = np.maximum(r.gamma(1.5, KT, int(th.sum())), E_MIN)
            if tally: ncs.append(nc[die])
            E = newE[~die]; nc = nc[~die]
            if tally:
                tF.append(tf[die]); tS.append(ts[die]); tT.append(tt[die])
                tf, ts, tt = tf[~die], ts[~die], tt[~die]
    p = fis/n_hist
    return dict(n=n_hist, fissions=fis, k=NU*p, sk=NU*np.sqrt(p*(1-p)/n_hist),
                coll=ncoll/n_hist, fE=np.concatenate(fE) if fE else np.zeros(0),
                nc=np.concatenate(ncs) if ncs else np.zeros(0),
                smap=smap, flux=flux, absE=absE, expF=expF, expC=expC,
                tF=np.concatenate(tF) if tF else np.zeros(0),
                tS=np.concatenate(tS) if tS else np.zeros(0),
                tT=np.concatenate(tT) if tT else np.zeros(0))

t0 = time.time(); res = run(1_000_000); dt = time.time()-t0
print(f"N = {res['n']:,} in {dt:.0f}s ({res['n']/dt:,.0f} neutrons/s)   fissions = {res['fissions']:,}")
print(f"k_inf = {res['k']:.4f} +/- {2*res['sk']:.4f} (2 sigma)")
print(f"collisions/history = {res['coll']:.1f}   cross-check vs analog loop: {slow['k']:.4f}")
print(f"\n1e9 neutrons on one core: that is {1e9/(res['n']/dt)/3600:.1f} hours at this rate.")
print(f"It would shrink the error bar to about +/-{2*res['sk']/np.sqrt(1000):.5f}, which changes no conclusion")
print(f"here -- the verdict is already {abs(res['k']-1)/res['sk']:.0f} standard errors from k=1.")
print("Set RUN_1E9 = True in the next cell if you want it anyway.")

In [ ]:
RUN_1E9 = False        # <- set True to run the billion-neutron case (see the estimate above)
if RUN_1E9:
    # 10^9 histories split into 20 independent chunks (own seeds), run in parallel on the CPU cores
    import multiprocessing as mp
    N_BIG, CHUNK = 1_000_000_000, 50_000_000
    NPROC = max(1, (os.cpu_count() or 2) - 2)
    def _chunk(j): return run(CHUNK, seed=SEED + 7919*(j+1), tally=False)["fissions"]
    t0 = time.time()
    with mp.get_context("fork").Pool(NPROC) as pool:
        fis_big = sum(pool.map(_chunk, range(N_BIG//CHUNK)))
    p = fis_big/N_BIG
    big = dict(n=N_BIG, fissions=fis_big, k=NU*p, sk=NU*np.sqrt(p*(1-p)/N_BIG))
    print(f"N = {big['n']:,} in {(time.time()-t0)/3600:.2f} h on {NPROC} processes   fissions = {fis_big:,}")
    print(f"k_inf = {big['k']:.5f} +/- {2*big['sk']:.5f} (2 sigma)")

### 12. Plot 3 — convergence
$$\sigma_k=\nu\sqrt{\frac{(f/N)(1-f/N)}{N}}$$

In [ ]:
Ns = np.array([1_000,3_000,10_000,30_000,100_000,300_000,1_000_000])
ks, es = [], []
for M in Ns:
    d = run(int(M), seed=SEED+M, tally=False); ks.append(d["k"]); es.append(d["sk"])
ks, es = np.array(ks), np.array(es)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].errorbar(Ns, ks, 2*es, fmt="o-", ms=4, capsize=3, lw=1.2)
ax[0].axhline(1.0, color="#d62728", lw=1.5, label="$k_\\infty=1$")
ax[0].set(xscale="log", xlabel="neutrons tracked $N$", ylabel="$k_\\infty$",
          ylim=(0,1.15), title="criticality margin")
ax[0].legend(fontsize=8)
ax[1].loglog(Ns, es/ks, "o-", ms=4, lw=1.2)
ax[1].loglog(Ns, (es/ks)[0]*np.sqrt(Ns[0]/Ns), "k--", lw=.8, label="$1/\\sqrt{N}$")
ax[1].set(xlabel="neutrons tracked $N$", ylabel="relative $\\sigma_k$"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
for M,k,e in zip(Ns,ks,es): print(f"N={M:>9,}   k_inf={k:.4f} +/- {2*e:.4f}   ({100*e/k:4.2f}%)")
print(f"\nk_inf = {ks[-1]:.4f} +/- {2*es[-1]:.4f};  "
      f"{abs(ks[-1]-1)/es[-1]:.0f} standard errors below critical.")

### 13. Plot 4 — why it fails
$$\eta(E)=\nu\,\frac{\sigma_f}{\sigma_f+\sigma_c}\quad\text{— neutrons returned per neutron absorbed}$$

In [ ]:
Sf = sum(COMP[q]*XS[q]["f"](Ep) for q in COMP)
Sc = sum(COMP[q]*XS[q]["c"](Ep) for q in COMP)
St = sum(COMP[q]*XS[q][k](Ep) for q in COMP for k in "eicf")
eta = NU*Sf/np.maximum(Sf+Sc,1e-30)

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].hist(np.log10(np.maximum(res["fE"],1e-10)), 80, histtype="stepfilled", alpha=.6, color="#9467bd")
ax[0].set(xlabel="$\\log_{10}E$ at fission (MeV)", ylabel="count",
          title=f"median {np.median(res['fE']):.2f} MeV -- fissions stay fast")
ax[1].hist(res["nc"], bins=60, histtype="stepfilled", alpha=.7, color="#2ca02c")
ax[1].set(yscale="log", xlabel="collisions", ylabel="histories",
          title=f"collisions/history (mean {res['coll']:.1f})")
ax[2].loglog(Ep, np.maximum(eta,1e-4), color="#9467bd", lw=1.4, label="$\\eta$")
ax[2].axhline(1, color="#d62728", ls="--", lw=1.3, label="$\\eta=1$")
ax[2].set(xlabel="E (MeV)", ylabel="$\\eta$", ylim=(1e-3,10),
          title="$\\eta<1$ across the slowing-down range"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

for E in (2.53e-8, 1e-6, 1e-4, 1e-2, 1.0, 2.0):
    f = sum(COMP[q]*XS[q]["f"](E)[0] for q in COMP)
    cc= sum(COMP[q]*XS[q]["c"](E)[0] for q in COMP)
    print(f"E={E:8.1e} MeV   eta = {NU*f/max(f+cc,1e-30):6.3f}")
Ebb = sample_birth(1_000_000, np.random.default_rng(3))
Stb = sum(COMP[q]*XS[q][k](Ebb) for q in COMP for k in "eicf")
Sfb = sum(COMP[q]*XS[q]["f"](Ebb) for q in COMP)
print(f"\nfirst-collision estimate nu*<sigma_f/sigma_t> = {NU*np.mean(Sfb/Stb):.4f}")
print(f"full simulation                              = {res['k']:.4f}")

### 14. Plot 5 — the slowing-down process
Left: energy of every neutron at each collision number (colour = share of the neutrons still alive at
that collision), with the median in white. Right: the neutron flux per unit lethargy, $E\,\phi(E)$,
from a collision estimator (each collision adds $1/\Sigma_t$ to its energy bin, equal on average to the track length).
Expected shape: the fission peak near 1–2 MeV; a roughly flat $1/E$ slowing-down plateau with dips at
the U-238 resonances when there is a moderator; a Maxwellian peak near 0.03–0.05 eV if neutrons thermalise.

In [ ]:
lo   = LE[:-1]; hi = LE[1:]; mid = .5*(lo+hi)
cmax = int(min(CMAX, max(10, np.percentile(res["nc"], 99))))
S    = res["smap"][:cmax]; alive = S.sum(1)
frac = S/np.maximum(alive[:,None], 1)
cdf  = np.cumsum(S, 1)/np.maximum(alive[:,None], 1)
med  = mid[np.minimum((cdf < .5).sum(1), mid.size-1)]
n    = np.arange(1, cmax+1)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
im = ax[0].pcolormesh(np.arange(.5, cmax+1)+.5, LE, np.ma.masked_equal(frac.T, 0), shading="flat",
                      cmap="viridis", norm=plt.matplotlib.colors.LogNorm(vmin=1e-4, vmax=1))
ax[0].plot(n+.5, med, "w-", lw=1.5, label="median energy")
ax[0].axhline(np.log10(1e-6), color="r", ls=":", lw=1, label="1 eV")
ax[0].set(xscale="log", xlabel="collision number", ylabel="$\\log_{10}E$ (MeV)", ylim=(-9, 1.3),
          title="energy vs collision number"); ax[0].legend(fontsize=7, loc="upper right")
plt.colorbar(im, ax=ax[0], label="share of live neutrons")

phi = res["flux"]/np.log(10**hi/10**lo)            # per unit lethargy
phi = phi/phi.max()
ax[1].step(10**mid, np.maximum(phi, 1e-6), where="mid", lw=1.2)
for e,l in ((1e-6,"1 eV"),(0.1,"0.1 MeV")): ax[1].axvline(e, color="grey", ls=":", lw=1)
for x,l in ((3e-9,"thermal"),(1e-3,"slowing down"),(1.5,"fast")):
    ax[1].text(x, 2.2, l, ha="center", fontsize=8)
ax[1].set(xscale="log", yscale="log", ylim=(1e-5, 4), xlim=(1e-9, 20), xlabel="E (MeV)",
          ylabel="$E\\,\\phi(E)$ (normalised)", title="flux per unit lethargy")
plt.tight_layout(); plt.show()
for c in (1, 5, 10, 20, 50, 100, 300, 1000):
    if c <= cmax: print(f"after {c:5d} collisions: median E = {10**med[c-1]:.3e} MeV, {alive[c-1]/res['n']:.1%} still alive")

### 15. Plot 6 — neutron lifetimes: fast, slowing down and thermal
Time is the one quantity that needs **absolute** densities: each flight takes $1/(\Sigma_t v)$, with
$v = 1.383\times10^{9}\sqrt{E\,[\text{MeV}]}$ cm/s. Uranium metal (19.05 g/cm³) and water
(H$_2$O 1.00, D$_2$O 1.105 g/cm³) are mixed with volumes additive. Groups: **fast** above 0.1 MeV,
**slowing down** 1 eV–0.1 MeV, **thermal** below 1 eV. $k_\infty$ does not depend on any of this.

In [ ]:
life = res["tF"] + res["tS"] + res["tT"]
mt   = np.array([res["tF"].mean(), res["tS"].mean(), res["tT"].mean()])
grp  = ["fast\n> 0.1 MeV", "slowing down\n1 eV - 0.1 MeV", "thermal\n< 1 eV"]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
bars = ax[0].bar(grp, np.maximum(mt, 1e-12), color=["#d62728", "#ff7f0e", "#1f77b4"])
for b_, v in zip(bars, mt): ax[0].text(b_.get_x()+b_.get_width()/2, max(v,2e-10)*1.3, f"{v:.2e} s" if v > 0 else "none", ha="center", fontsize=8)
ax[0].set(yscale="log", ylim=(1e-10, 1e-1), ylabel="mean time per neutron (s)", title="where a neutron spends its life")
bins = np.logspace(-10, 0, 120)
for t_,c_,l_ in ((res["tF"],"#d62728","fast"),(res["tS"],"#ff7f0e","slowing down"),(res["tT"],"#1f77b4","thermal"),(life,"k","total")):
    if (t_>0).any(): ax[1].hist(t_[t_>0], bins, histtype="step", lw=1.3, color=c_, label=l_)
ax[1].set(xscale="log", yscale="log", xlabel="time (s)", ylabel="neutrons",
          title=f"lifetime distribution (mean {life.mean():.2e} s)"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
print(f"number density scale: {n_scale(COMP):.4e} atoms/(b cm) per unit of COMP")
for g,v in zip(("fast", "slowing down", "thermal"), mt): print(f"mean time {g:13s}: {v:.3e} s  ({v/life.mean():6.1%} of the lifetime)")
print(f"mean neutron lifetime     : {life.mean():.3e} s")
print(f"neutrons that reach thermal (< 1 eV): {np.mean(res['tT'] > 0):.1%}")

### 16. Plot 7 — absorption in moderator vs fuel, and the fission-to-capture ratio
Left: what each source neutron is finally absorbed by. Right: fissions per capture in the **fuel**
as a function of energy: the sampled Monte Carlo events (points) against the ratio the cross sections
predict, $\sum N\sigma_f/\sum N\sigma_c$ for the uranium, averaged over the same collisions in each energy
bin (line). Agreement means the sampling is doing what the cross sections say.

In [ ]:
A    = res["absE"]; N = res["n"]; mid = .5*(LE[:-1]+LE[1:])
fuel = [q for q in COMP if q in ("U5", "U8")]; mod = [q for q in COMP if q not in fuel]
cap  = {q: A[q,"c"].sum()/N for q in COMP}; fis_ = {q: A[q,"f"].sum()/N for q in COMP}
NAME = {"U5":"U-235", "U8":"U-238", "H":"H-1", "D":"D-2", "O":"O-16"}

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
x = np.arange(len(COMP))
ax[0].bar(x, [cap[q] for q in COMP], color="#7f7f7f", label="capture")
ax[0].bar(x, [fis_[q] for q in COMP], bottom=[cap[q] for q in COMP], color="#9467bd", label="fission")
for i,q in enumerate(COMP): ax[0].text(i, cap[q]+fis_[q]+.01, f"{cap[q]+fis_[q]:.3f}", ha="center", fontsize=8)
af = sum(cap[q]+fis_[q] for q in fuel); am = sum(cap[q] for q in mod)
ax[0].set(xticks=x, xticklabels=[NAME[q] for q in COMP], ylabel="absorptions per source neutron", ylim=(0, 1.12),
          title=f"fuel {af:.1%}  vs  moderator {am:.1%}"); ax[0].legend(fontsize=7)

fF = sum(A[q,"f"] for q in fuel); cF = sum(A[q,"c"] for q in fuel); ok_ = (fF >= 10) & (cF >= 10)
ex_ = (res["expC"] > 0) & (res["expF"] > 0)
ax[1].step(10**mid[ex_], (res["expF"]/np.maximum(res["expC"],1e-300))[ex_], where="mid", color="#7f7f7f", lw=1,
           label="expected from cross sections $\\Sigma_f/\\Sigma_c$ (fuel)")
rat = fF/np.maximum(cF,1)
ax[1].errorbar(10**mid[ok_], rat[ok_], yerr=(rat*np.sqrt(1/np.maximum(fF,1)+1/np.maximum(cF,1)))[ok_],
               fmt="o", ms=3, lw=.8, color="#9467bd", label="Monte Carlo tally (bins with >= 10 of each)")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set(xlabel="E (MeV)", ylabel="fissions / captures in fuel", ylim=(1e-6, 1e3),
          title=f"fission/capture: fuel {res['fissions']/max(sum(cap[q] for q in fuel)*N,1):.3f}, all {res['fissions']/max(sum(cap.values())*N,1):.3f}"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
for q in COMP: print(f"{NAME[q]:6s} capture {cap[q]:.4f}   fission {fis_[q]:.4f}   per source neutron")
print(f"absorbed in fuel {af:.4f}   in moderator {am:.4f}")
print(f"fission-to-capture ratio (all captures) = {res['fissions']/max(sum(cap.values())*N,1):.4f}")
print(f"fission-to-capture ratio (fuel only)    = {res['fissions']/max(sum(cap[q] for q in fuel)*N,1):.4f}")

### 17. Sensitivity — does the grid resolution change the answer?
The 50-point grid is only a *minimum* resolution. This re-runs the whole simulation on
cross sections resampled onto only those 50 points, to show what that coarsening costs.

In [ ]:
XS_FULL = {q:dict(XS[q]) for q in XS}
def use_grid(coarse):
    for q in XS:
        for k in "eicf":
            if coarse:
                v = XS_FULL[q][k](E_GRID)
                XS[q][k] = XSec(E_GRID, v) if (v>0).any() else ZERO
            else:
                XS[q][k] = XS_FULL[q][k]

use_grid(True);  kc = run(200_000, seed=SEED+5, tally=False)
use_grid(False); kf = run(200_000, seed=SEED+5, tally=False)
print(f"full pointwise data : k_inf = {kf['k']:.4f} +/- {2*kf['sk']:.4f}")
print(f"50-point grid only  : k_inf = {kc['k']:.4f} +/- {2*kc['sk']:.4f}   ({100*(kc['k']-kf['k'])/kf['k']:+.1f}%)")
print("\nFive points per decade cannot resolve the U-238 capture resonances, so the coarse")
print("grid mis-states resonance absorption. The reported number uses the full pointwise data.")

### 18. Conclusion

In [ ]:
N_HIST = 1_000_000
L = ["",
 f"k_inf(pure U-238, infinite) = {ks[-1]:.3f} +/- {2*es[-1]:.3f}   ->  CANNOT GO CRITICAL",
 f"{abs(ks[-1]-1)/es[-1]:.0f} standard errors below 1. Statistics are not the limiting factor.",
 "",
 f"Each of the {N_HIST:,} histories starts with one neutron born with energy sampled from s(E).",
 "U-238 fissions only above a threshold of about 1 MeV; below it, fission is negligible next to capture.",
 f"Neutrons are born averaging 1.93 MeV, so {(Eb>1).mean():.0%} start above that threshold, but every collision",
 "moves them down and nothing moves them back up. Inelastic scattering removes several hundred keV",
 "at a time and beats fission roughly 5:1 in the window where fission is possible.",
 "",
 "Below the threshold the only thing left is capture, so eta = nu*sf/(sf+sc) collapses.",
 f"Median energy at fission is {np.median(res['fE']):.2f} MeV, and a history lasts {res['coll']:.0f} collisions",
 "before it ends in capture.",
 "",
 "No amount of statistics or reactor size changes this: the medium is infinite already,",
 "so there is no leakage to remove and nothing left to optimise.",
]
print("\n".join(L))